# Figure 2 - Robustness and Null Analysis of Yeast Annotations

Robustness and null analysis of Gene Ontology Biological Process annotations in the yeast genetic interaction profile similarity matrix. Panels A/B test sensitivity to dendrogram distance threshold and minimum cluster size by tracking recovery of the seven headline GO BP terms from the Fig. 1B parent-level reference analysis after rerunning enrichment testing. Panels C/D compare the observed 331 significant cluster-term pairs against two 1,000-replicate null models: same-size random clusters with annotations fixed and annotation-label permutations with clusters fixed.


## Setup

Enable inline plotting and define the figure export paths.

In [ ]:
# Enable inline plotting for notebooks
%matplotlib inline

In [ ]:
# Figure export paths and utilities (PNG, 300 DPI).
from pathlib import Path
import warnings

import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message=r"Dropped .* annotations after matrix filtering.*",
    category=RuntimeWarning,
)

NB_ID = "fig_2"
PNG_DIR = Path("png") / NB_ID
PNG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure_png(name, *, fig=None, dpi=300, pad_inches=0.02):
    """Save a Matplotlib figure as a high-quality PNG in the notebook export folder."""
    if fig is None:
        fig = plt.gcf()
    output_path = PNG_DIR / f"{name}.png"
    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=pad_inches,
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    print(f"Saved {output_path}")
    return output_path

## Load Inputs

Load GO Biological Process gene sets and the yeast genetic interaction profile similarity matrix.


In [ ]:
import json

import pandas as pd

DATA_DIR = Path("data")
GO_BP_PATH = DATA_DIR / "go_bp_name_to_orfs.json"
MATRIX_PATH = DATA_DIR / "gi_pcc_sampled.tsv"

with GO_BP_PATH.open("r", encoding="utf-8") as fh:
    go_bp = json.load(fh)

DF_GI_PCC = pd.read_csv(MATRIX_PATH, sep="\t", index_col=0)

print(f"GO BP terms loaded: {len(go_bp):,}")
print(f"Matrix shape: {DF_GI_PCC.shape[0]:,} × {DF_GI_PCC.shape[1]:,}")
print(f"Row/column labels identical: {DF_GI_PCC.index.equals(DF_GI_PCC.columns)}")

## Run Reference Clustering

Cluster the full yeast genetic interaction profile similarity matrix and test GO Biological Process enrichment using the fixed reference settings: Ward/Euclidean linkage, distance threshold 16, minimum cluster size 30, minimum overlap 2, global Benjamini-Hochberg FDR, and `qval <= 0.05`. This rerun uses the same reference configuration as `fig_1.ipynb` and `supp_fig_1.ipynb`; the sweeps and null analyses below all reuse this single reference clustering result.


In [ ]:
import himalayas
from himalayas import Analysis, Annotations, Matrix

print(f"HiMaLAYAS version: {himalayas.__version__}")

REFERENCE_CONFIG = dict(
    linkage_method="ward",
    linkage_metric="euclidean",
    linkage_threshold=16,
    optimal_ordering=True,
    min_cluster_size=30,
)
MIN_OVERLAP = 2
QVAL_CUTOFF = 0.05
FDR_SCOPE = "global"

matrix = Matrix(DF_GI_PCC)
annotations = Annotations(go_bp, matrix)

analysis = (
    Analysis(matrix, annotations)
    .cluster(**REFERENCE_CONFIG)
    .enrich(min_overlap=MIN_OVERLAP)
    .finalize(col_cluster=True, fdr_scope=FDR_SCOPE)
)
results = analysis.results
results_sig = results.filter(f"qval <= {QVAL_CUTOFF}")

assert (len(results.df), len(results_sig.df), len(results.clusters.cluster_sizes)) == (
    709,
    331,
    7,
), "Genetic interaction profile similarity reference clustering result changed from expected publication values"

REFERENCE_N_CLUSTERS = len(results.clusters.cluster_sizes)
n_observed_sig = len(results_sig.df)

print(f"All enriched rows: {len(results.df):,}")
print(f"Significant cluster-term pairs (q<={QVAL_CUTOFF}): {n_observed_sig:,}")
print(f"Clusters: {REFERENCE_N_CLUSTERS}")

## Run Threshold and Minimum-Cluster-Size Sensitivity Sweeps

Rerun clustering and enrichment at each dendrogram distance threshold in `THRESHOLD_GRID` and each minimum cluster size in `MIN_SIZE_GRID`, holding all other reference settings fixed. For each swept clustering, a reference cluster's headline GO BP term counts as recovered if the sweep cluster with greatest gene overlap by Jaccard index has the term significant at `qval <= 0.05`.


In [ ]:
THRESHOLD_GRID = [12, 14, 15, 16, 17, 18, 20, 22]
MIN_SIZE_GRID = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100]

# Reference headline GO term per reference cluster, and each cluster's ORF set
# (label_to_cluster maps each ORF to its cluster id for this clustering run).
reference_cluster_labels = results_sig.cluster_labels(rank_by="q", label_mode="top_term")
reference_label_to_cluster = results.clusters.label_to_cluster
print(
    f"Reference label count vs matrix row count: "
    f"{len(reference_label_to_cluster)} / {len(DF_GI_PCC.index)}"
)

reference_cluster_orf_sets = {
    int(cid): {orf for orf, orf_cid in reference_label_to_cluster.items() if orf_cid == cid}
    for cid in reference_cluster_labels["cluster"]
}


def best_jaccard_match(reference_orf_set, sweep_label_to_cluster):
    """Return the sweep cluster id with greatest Jaccard overlap to reference_orf_set."""
    sweep_cluster_orfs = {}
    for orf, cid in sweep_label_to_cluster.items():
        sweep_cluster_orfs.setdefault(cid, set()).add(orf)

    best_cid, best_jaccard = None, -1.0
    for cid, sweep_orf_set in sweep_cluster_orfs.items():
        union = len(reference_orf_set | sweep_orf_set)
        jaccard = len(reference_orf_set & sweep_orf_set) / union if union else 0.0
        if jaccard > best_jaccard:
            best_cid, best_jaccard = cid, jaccard
    return best_cid


def run_sensitivity_sweep(param_name, param_grid):
    rows = []
    for value in param_grid:
        sweep_config = dict(REFERENCE_CONFIG)
        sweep_config[param_name] = value
        sweep_analysis = (
            Analysis(matrix, annotations)
            .cluster(**sweep_config)
            .enrich(min_overlap=MIN_OVERLAP)
            .finalize(col_cluster=True)
        )
        sweep_results = sweep_analysis.results
        sweep_results_sig = sweep_results.filter(f"qval <= {QVAL_CUTOFF}")
        sweep_label_to_cluster = sweep_results.clusters.label_to_cluster

        n_recovered = 0
        for _, ref_row in reference_cluster_labels.iterrows():
            ref_term = ref_row["label"]
            ref_orf_set = reference_cluster_orf_sets[int(ref_row["cluster"])]
            matched_cid = best_jaccard_match(ref_orf_set, sweep_label_to_cluster)
            matched_sig_terms = set(
                sweep_results_sig.df.loc[sweep_results_sig.df["cluster"] == matched_cid, "term"]
            )
            if ref_term in matched_sig_terms:
                n_recovered += 1

        rows.append(
            {
                param_name: value,
                "n_clusters": len(sweep_results.clusters.cluster_sizes),
                "sig_rows": len(sweep_results_sig.df),
                "top_term_retention_rate": n_recovered / REFERENCE_N_CLUSTERS,
            }
        )
    return pd.DataFrame(rows)


threshold_sensitivity = run_sensitivity_sweep("linkage_threshold", THRESHOLD_GRID)
threshold_sensitivity = threshold_sensitivity.rename(columns={"linkage_threshold": "threshold"})

min_size_sensitivity = run_sensitivity_sweep("min_cluster_size", MIN_SIZE_GRID)

print(f"\nSweep runs completed: {len(threshold_sensitivity) + len(min_size_sensitivity)}")
print("\nThreshold sensitivity:")
print(threshold_sensitivity.to_string(index=False))
print("\nMin-cluster-size sensitivity:")
print(min_size_sensitivity.to_string(index=False))

## Export Individual Panels

Export Panels A and B as standalone PNGs for manual figure assembly in PowerPoint.

In [ ]:
LABEL_COLOR = "black"
FONT = "Helvetica"
plt.rcParams["font.family"] = FONT
COUNT_COLOR = "#4d4d4d"
RETENTION_COLOR = "#d73027"
WEAKEST_COLOR = "#e6550d"
STRONGEST_COLOR = "#787878"
REFERENCE_GRAY = "#787878"
PANEL_FIGSIZE = (10, 5)

TITLE_FONTSIZE = 26
TITLE_PAD = 22
AXIS_LABEL_FONTSIZE = 20
TICK_LABEL_FONTSIZE = 18
INSET_TEXT_FONTSIZE = 20
REFERENCE_TEXT_FONTSIZE = 18
LEGEND_FONTSIZE = 18
LINEWIDTH = 3
MARKER_SIZE = 10
GRID_COLOR = "#cecece"
GRID_LINEWIDTH = 1
GRID_ALPHA = 0.8
REFERENCE_LINEWIDTH = 1.2
REFERENCE_ALPHA = 0.8

In [ ]:
import numpy as np

fig_b, ax_b = plt.subplots(figsize=PANEL_FIGSIZE)
fig_b.patch.set_facecolor("white")
ax_b.set_facecolor("white")

ax_b2 = ax_b.twinx()
retained_labels_b = threshold_sensitivity["top_term_retention_rate"] * REFERENCE_N_CLUSTERS
ax_b.plot(
    threshold_sensitivity["threshold"],
    retained_labels_b,
    marker="o",
    markersize=MARKER_SIZE,
    linewidth=LINEWIDTH,
    color=RETENTION_COLOR,
    label="Original GO terms\nrecovered",
)
ax_b2.plot(
    threshold_sensitivity["threshold"],
    threshold_sensitivity["n_clusters"],
    marker="s",
    markersize=MARKER_SIZE,
    linewidth=LINEWIDTH,
    color=COUNT_COLOR,
    label="Clusters produced",
)
ax_b.axvline(
    16, color=REFERENCE_GRAY, linewidth=REFERENCE_LINEWIDTH, linestyle="--", alpha=REFERENCE_ALPHA
)
ax_b.set_xlabel(
    "Distance threshold", fontsize=AXIS_LABEL_FONTSIZE, color=LABEL_COLOR, fontname=FONT
)
ax_b.set_ylabel(
    "Original GO terms\nrecovered (of 7)",
    fontsize=AXIS_LABEL_FONTSIZE,
    color=RETENTION_COLOR,
    fontname=FONT,
)
ax_b.set_ylim(3.3, 7.25)
ax_b.set_yticks([4, 5, 6, 7])
ax_b.text(
    16.1,
    ax_b.get_ylim()[0] + 0.75,
    "reference = 16",
    fontsize=REFERENCE_TEXT_FONTSIZE,
    color=REFERENCE_GRAY,
    fontname=FONT,
)
ax_b2.set_ylabel(
    "Clusters produced", fontsize=AXIS_LABEL_FONTSIZE, color=COUNT_COLOR, fontname=FONT
)
ymin_b2, ymax_b2 = ax_b2.get_ylim()
ax_b2.set_yticks(range(int(np.floor(ymin_b2)), int(np.ceil(ymax_b2)) + 1))
ax_b.set_title(
    "Distance-threshold sensitivity",
    fontsize=TITLE_FONTSIZE,
    color=LABEL_COLOR,
    fontname=FONT,
    loc="left",
    pad=TITLE_PAD,
)
ax_b.tick_params(axis="x", labelsize=TICK_LABEL_FONTSIZE)
ax_b.tick_params(axis="y", colors=RETENTION_COLOR, labelsize=TICK_LABEL_FONTSIZE)
ax_b2.tick_params(axis="y", colors=COUNT_COLOR, labelsize=TICK_LABEL_FONTSIZE)
ax_b.grid(axis="y", color=GRID_COLOR, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)
lines1_b, labels1_b = ax_b.get_legend_handles_labels()
lines2_b, labels2_b = ax_b2.get_legend_handles_labels()
ax_b.legend(
    lines1_b + lines2_b,
    labels1_b + labels2_b,
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    facecolor="white",
    framealpha=1.0,
    loc="lower left",
    labelspacing=0.6,
)
ax_b.spines["top"].set_visible(False)
ax_b2.spines["top"].set_visible(False)

fig_b.tight_layout()
save_figure_png("panel_a_threshold_sensitivity", fig=fig_b)
plt.show()

In [ ]:
fig_c, ax_c = plt.subplots(figsize=PANEL_FIGSIZE)
fig_c.patch.set_facecolor("white")
ax_c.set_facecolor("white")

ax_c2 = ax_c.twinx()
retained_labels_c = min_size_sensitivity["top_term_retention_rate"] * REFERENCE_N_CLUSTERS
ax_c.plot(
    min_size_sensitivity["min_cluster_size"],
    retained_labels_c,
    marker="o",
    markersize=MARKER_SIZE,
    linewidth=LINEWIDTH,
    color=RETENTION_COLOR,
    label="Original GO terms\nrecovered",
)
ax_c2.plot(
    min_size_sensitivity["min_cluster_size"],
    min_size_sensitivity["n_clusters"],
    marker="s",
    markersize=MARKER_SIZE,
    linewidth=LINEWIDTH,
    color=COUNT_COLOR,
    label="Clusters produced",
)
ax_c.axvline(
    30, color=REFERENCE_GRAY, linewidth=REFERENCE_LINEWIDTH, linestyle="--", alpha=REFERENCE_ALPHA
)
ax_c.set_xlabel(
    "Minimum cluster size", fontsize=AXIS_LABEL_FONTSIZE, color=LABEL_COLOR, fontname=FONT
)
ax_c.set_ylabel(
    "Original GO terms\nrecovered (of 7)",
    fontsize=AXIS_LABEL_FONTSIZE,
    color=RETENTION_COLOR,
    fontname=FONT,
)
ax_c.set_ylim(3.3, 7.25)
ax_c.set_yticks([4, 5, 6, 7])
ax_c.text(
    31,
    ax_c.get_ylim()[0] + 1.75,
    "reference = 30",
    fontsize=REFERENCE_TEXT_FONTSIZE,
    color=REFERENCE_GRAY,
    fontname=FONT,
)
ax_c2.set_ylabel(
    "Clusters produced", fontsize=AXIS_LABEL_FONTSIZE, color=COUNT_COLOR, fontname=FONT
)
ymin_c2, ymax_c2 = ax_c2.get_ylim()
ax_c2.set_yticks(range(int(np.floor(ymin_c2)), int(np.ceil(ymax_c2)) + 1))
ax_c.set_title(
    "Minimum-cluster-size sensitivity",
    fontsize=TITLE_FONTSIZE,
    color=LABEL_COLOR,
    fontname=FONT,
    loc="left",
    pad=TITLE_PAD,
)
ax_c.tick_params(axis="x", labelsize=TICK_LABEL_FONTSIZE)
ax_c.tick_params(axis="y", colors=RETENTION_COLOR, labelsize=TICK_LABEL_FONTSIZE)
ax_c2.tick_params(axis="y", colors=COUNT_COLOR, labelsize=TICK_LABEL_FONTSIZE)
ax_c.grid(axis="y", color=GRID_COLOR, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)
lines1_c, labels1_c = ax_c.get_legend_handles_labels()
lines2_c, labels2_c = ax_c2.get_legend_handles_labels()
ax_c.legend(
    lines1_c + lines2_c,
    labels1_c + labels2_c,
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    facecolor="white",
    framealpha=1.0,
    loc="lower left",
    labelspacing=0.6,
)
ax_c.spines["top"].set_visible(False)
ax_c2.spines["top"].set_visible(False)

fig_c.tight_layout()
save_figure_png("panel_b_min_cluster_size_sensitivity", fig=fig_c)
plt.show()

## Same-Size Random Clusters With Annotations Fixed

Keep the GO Biological Process annotations fixed. For each of `N_REPLICATES` replicates, randomly partition the matrix gene universe into clusters that preserve the observed cluster count and sizes (`results.clusters.cluster_sizes`), without rerunning hierarchical clustering. Rerun enrichment against the fixed annotations, followed by global Benjamini-Hochberg FDR, exactly as in the annotation-label permutation analysis below.


In [ ]:
from types import SimpleNamespace

from himalayas.core.enrichment import run_cluster_hypergeom

N_REPLICATES = 1000
RANDOM_CLUSTER_SEED = 20260810

gene_universe = matrix.labels
rand_rng = np.random.default_rng(RANDOM_CLUSTER_SEED)
cluster_sizes = results.clusters.cluster_sizes
cluster_ids_ordered = list(cluster_sizes.keys())

rand_counts = []
for _ in range(N_REPLICATES):
    shuffled = rand_rng.permutation(gene_universe)
    cluster_to_labels = {}
    offset = 0
    for cid in cluster_ids_ordered:
        size = cluster_sizes[cid]
        cluster_to_labels[cid] = set(shuffled[offset : offset + size])
        offset += size

    rand_clusters = SimpleNamespace(
        unique_clusters=np.array(cluster_ids_ordered),
        cluster_to_labels=cluster_to_labels,
        cluster_sizes=dict(cluster_sizes),
        merge_small_clusters=results.clusters.merge_small_clusters,
        min_cluster_size=results.clusters.min_cluster_size,
    )

    rand_null_results = run_cluster_hypergeom(
        matrix, rand_clusters, annotations, min_overlap=MIN_OVERLAP
    ).with_qvalues(method=FDR_SCOPE)
    n_sig = (
        0
        if rand_null_results.df.empty
        else int((rand_null_results.df["qval"] <= QVAL_CUTOFF).sum())
    )
    rand_counts.append(n_sig)

rand_null_dist = pd.DataFrame({"n_sig": rand_counts})
rand_null_max = int(rand_null_dist["n_sig"].max())
rand_empirical_p = (int((rand_null_dist["n_sig"].to_numpy() >= n_observed_sig).sum()) + 1) / (
    N_REPLICATES + 1
)
rand_row = {
    "observed_value": n_observed_sig,
    "null_max": rand_null_max,
    "empirical_pvalue": rand_empirical_p,
}

print(f"Same-size random-cluster null ({N_REPLICATES} replicates, seed={RANDOM_CLUSTER_SEED}):")
print(f"  observed significant cluster-term pairs : {rand_row['observed_value']}")
print(f"  null max                                 : {rand_row['null_max']}")
print(f"  empirical p-value                        : {rand_row['empirical_pvalue']:.6f}")

## Annotation-Label Permutations With Clusters Fixed

Keep the observed reference clustering fixed. For each of `N_REPLICATES` replicates, apply a random bijection over the matrix gene universe to relabel which genes carry which GO Biological Process term membership, preserving every term's size exactly. Rebuild `Annotations` from the permuted mapping and rerun enrichment (`run_cluster_hypergeom`) against the fixed reference clusters, followed by global Benjamini-Hochberg FDR (`with_qvalues(method=FDR_SCOPE)`). Clustering is not rerun; only enrichment reruns per replicate.


In [ ]:
PERMUTATION_SEED = 20260809

# Prefilter GO-BP annotations to the matrix gene universe before permutation.
matrix_gene_set = set(matrix.labels)
go_bp_matrix = {term: [g for g in genes if g in matrix_gene_set] for term, genes in go_bp.items()}

perm_rng = np.random.default_rng(PERMUTATION_SEED)

perm_counts = []
for _ in range(N_REPLICATES):
    shuffled = perm_rng.permutation(gene_universe)
    relabel_map = dict(zip(gene_universe, shuffled))
    permuted_go_bp = {term: [relabel_map[g] for g in genes] for term, genes in go_bp_matrix.items()}
    perm_annotations = Annotations(permuted_go_bp, matrix)
    perm_null_results = run_cluster_hypergeom(
        matrix, results.clusters, perm_annotations, min_overlap=MIN_OVERLAP
    ).with_qvalues(method=FDR_SCOPE)
    n_sig = (
        0
        if perm_null_results.df.empty
        else int((perm_null_results.df["qval"] <= QVAL_CUTOFF).sum())
    )
    perm_counts.append(n_sig)

perm_null_dist = pd.DataFrame({"n_sig": perm_counts})
perm_null_max = int(perm_null_dist["n_sig"].max())
perm_empirical_p = (int((perm_null_dist["n_sig"].to_numpy() >= n_observed_sig).sum()) + 1) / (
    N_REPLICATES + 1
)
perm_row = {
    "observed_value": n_observed_sig,
    "null_max": perm_null_max,
    "empirical_pvalue": perm_empirical_p,
}

print(f"Annotation-label permutation null ({N_REPLICATES} replicates, seed={PERMUTATION_SEED}):")
print(f"  observed significant cluster-term pairs : {perm_row['observed_value']}")
print(f"  null max                                 : {perm_row['null_max']}")
print(f"  empirical p-value                        : {perm_row['empirical_pvalue']:.6f}")

## Export Null-Analysis Panels

Export Panels C and D as standalone PNGs for manual figure assembly in PowerPoint. Each panel is a compact null histogram with the observed value shown on the same transformed x-axis. The x positions use `log10(significant_pairs + 1)`, which preserves the zero-valued null replicates while making the distance from the null range to the observed value visible without an axis break.


In [ ]:
OBSERVED_COLOR = "#d73027"
NULL_COLOR = "#4d4d4d"
BAR_EDGE = "black"
SPINE_COLOR = "#333333"

TICK_PARAMS_LABELSIZE = 18
BAR_LABEL_FONTSIZE = 18
OBSERVED_LABEL_FONTSIZE = 20
STATS_LABEL_FONTSIZE = 18

BAR_WIDTH = 0.1
BAR_LINEWIDTH = 1
BAR_LABEL_Y_MULT = 1.15
OBSERVED_LINEWIDTH = 2
OBSERVED_LABEL_X_OFFSET = 0.1
ANNOTATION_ANCHOR_Y = 300
ANNOTATION_LINE_GAP_PT = 8
SPINE_LINEWIDTH = 0.5

XLIM_LEFT = -0.15
XLIM_RIGHT_PAD = 0.15
YLIM_BOTTOM = 0.8
YLIM_TOP = 1500

null_panels = [
    {
        "panel": "C",
        "name": "panel_c_random_cluster_null",
        "title": "Same-size random clusters",
        "dist": rand_null_dist,
        "observed": int(rand_row["observed_value"]),
        "null_max": int(rand_row["null_max"]),
        "empirical_p": float(rand_row["empirical_pvalue"]),
        "x_values": sorted(rand_null_dist["n_sig"].unique().tolist()),
    },
    {
        "panel": "D",
        "name": "panel_d_permutation_null",
        "title": "Annotation-label permutation",
        "dist": perm_null_dist,
        "observed": int(perm_row["observed_value"]),
        "null_max": int(perm_row["null_max"]),
        "empirical_p": float(perm_row["empirical_pvalue"]),
        "x_values": sorted(perm_null_dist["n_sig"].unique().tolist()),
    },
]

assert all(panel["observed"] == n_observed_sig for panel in null_panels)

x_tick_values = sorted(set([0, 1, 2, 10, 100, n_observed_sig]))


def x_transform(values):
    values = np.asarray(values, dtype=float)
    return np.log10(values + 1.0)


def plot_null_panel(ax, panel):
    """Render one null-histogram panel (bars + observed marker) onto ax."""
    counts = (
        panel["dist"]["n_sig"].value_counts().reindex(panel["x_values"], fill_value=0).sort_index()
    )
    x_raw = counts.index.to_numpy()
    x_pos = x_transform(x_raw)
    y = counts.to_numpy()

    ax.bar(
        x_pos,
        y,
        width=BAR_WIDTH,
        color=NULL_COLOR,
        edgecolor=BAR_EDGE,
        linewidth=BAR_LINEWIDTH,
        align="center",
        zorder=3,
    )
    for xi, yi in zip(x_pos, y):
        ax.text(
            xi,
            yi * BAR_LABEL_Y_MULT,
            f"{yi}",
            ha="center",
            va="bottom",
            fontsize=BAR_LABEL_FONTSIZE,
            color=LABEL_COLOR,
            fontname=FONT,
        )

    obs_x = float(x_transform([panel["observed"]])[0])
    ax.axvline(obs_x, color=OBSERVED_COLOR, linewidth=OBSERVED_LINEWIDTH, zorder=4)

    # One right-aligned annotation block anchored near the observed line: the
    # "Observed" line and the "largest null / p" line are stacked by a fixed
    # point-space offset (not data-space) so they read as one statistical
    # summary regardless of where they land on the log y-axis.
    anchor_xy = (obs_x - OBSERVED_LABEL_X_OFFSET, ANNOTATION_ANCHOR_Y)
    ax.annotate(
        f"Observed = {panel['observed']}",
        xy=anchor_xy,
        xycoords="data",
        ha="right",
        va="bottom",
        fontsize=OBSERVED_LABEL_FONTSIZE,
        color=OBSERVED_COLOR,
        fontname=FONT,
    )
    ax.annotate(
        f"Largest null = {panel['null_max']}; empirical $p$ = {panel['empirical_p']:.3f}",
        xy=anchor_xy,
        xytext=(0, -ANNOTATION_LINE_GAP_PT),
        textcoords="offset points",
        xycoords="data",
        ha="right",
        va="top",
        fontsize=STATS_LABEL_FONTSIZE,
        color=LABEL_COLOR,
        fontname=FONT,
    )

    ax.set_title(
        panel["title"],
        loc="left",
        fontsize=TITLE_FONTSIZE,
        fontname=FONT,
        color=LABEL_COLOR,
        pad=TITLE_PAD,
    )
    ax.set_xlabel(
        "Significant cluster-term pairs per replicate", fontsize=AXIS_LABEL_FONTSIZE, fontname=FONT
    )
    ax.set_ylabel("Null replicates (log scale)", fontsize=AXIS_LABEL_FONTSIZE, fontname=FONT)
    ax.set_xticks(x_transform(x_tick_values))
    ax.set_xticklabels([str(v) for v in x_tick_values], fontsize=TICK_LABEL_FONTSIZE, fontname=FONT)
    ax.set_xlim(XLIM_LEFT, obs_x + XLIM_RIGHT_PAD)
    ax.set_yscale("log")
    ax.set_ylim(YLIM_BOTTOM, YLIM_TOP)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA, which="major")
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(SPINE_COLOR)
    ax.spines["bottom"].set_color(SPINE_COLOR)
    ax.spines["left"].set_linewidth(SPINE_LINEWIDTH)
    ax.spines["bottom"].set_linewidth(SPINE_LINEWIDTH)
    ax.tick_params(axis="both", labelsize=TICK_PARAMS_LABELSIZE)

In [ ]:
fig_null_c, ax_null_c = plt.subplots(figsize=PANEL_FIGSIZE)
fig_null_c.patch.set_facecolor("white")

plot_null_panel(ax_null_c, null_panels[0])

fig_null_c.tight_layout()
save_figure_png("panel_c_random_cluster_null", fig=fig_null_c)
plt.show()

In [ ]:
fig_null_d, ax_null_d = plt.subplots(figsize=PANEL_FIGSIZE)
fig_null_d.patch.set_facecolor("white")

plot_null_panel(ax_null_d, null_panels[1])

fig_null_d.tight_layout()
save_figure_png("panel_d_permutation_null", fig=fig_null_d)
plt.show()

In [ ]:
# Computed numeric summary for the caption below.
rand_value_counts = rand_null_dist["n_sig"].value_counts().sort_index()
perm_value_counts = perm_null_dist["n_sig"].value_counts().sort_index()

print(f"Observed significant cluster-term pairs (q<={QVAL_CUTOFF}): {n_observed_sig}")

print(f"\nSame-size random-cluster null ({N_REPLICATES} replicates, seed={RANDOM_CLUSTER_SEED}):")
for n_sig, count in rand_value_counts.items():
    print(f"  {count} replicates produced {n_sig} significant pair(s)")
print(f"  null max: {rand_row['null_max']}")
print(f"  empirical p-value: {rand_row['empirical_pvalue']:.6f}")

print(f"\nAnnotation-label permutation null ({N_REPLICATES} replicates, seed={PERMUTATION_SEED}):")
for n_sig, count in perm_value_counts.items():
    print(f"  {count} replicates produced {n_sig} significant pair(s)")
print(f"  null max: {perm_row['null_max']}")
print(f"  empirical p-value: {perm_row['empirical_pvalue']:.6f}")

## Caption / Claim Boundary

**Figure 2. Robustness and null analysis of yeast annotations.** Panels A/B test sensitivity of the Fig. 1B parent-level reference analysis to dendrogram distance threshold and minimum cluster size. Red curves show recovery of the seven headline GO BP terms from that reference analysis after rerunning enrichment testing; gray curves show the number of clusters produced at each setting. These panels test annotation recovery, not invariant cluster membership or one-to-one cluster identity.

Panels C/D compare the observed 331 significant cluster-term pairs (`qval <= 0.05`) against two 1,000-replicate null models: same-size random clusters with annotations fixed and annotation-label permutations with clusters fixed. Each replicate uses the same enrichment test and FDR correction as the reference analysis. These null analyses test whether the observed enrichment signal is reproduced by arbitrary same-size cluster partitions or arbitrary annotation labels; they do not claim optimal clustering, biological completeness, or superiority over alternative clustering methods.
